# Signals and Systems Final Project

### Miguel Posada, James Pitts, and Maddox Nichols

The code below includes a setup block and then three separate filters. Each one runs separately so just run setup then pick any filter block to run.

- [Band Pass](#band_pass)
- [Low Pass](#low_pass)
- [High Pass](#high_pass)

Additionally the low and high pass are run through a band pass first to remove extra harmonics and some other random noise that we were picking up during testing


### Import Libraries and Define Parameters

In [ ]:
import numpy as np
import sounddevice as sd

#default config stuff that shouldn't need to be changed
fs = 44100          
blocksize = 2048    
input_device = None  
output_device = None 

In [ ]:
import sounddevice as sd; print(sd.query_devices())

<a id='band_pass'></a>
## Band Pass 

In [ ]:
f_low = 80    # High-Pass Cutoff (Only allow stuff past this)
f_high = 1500   # Low-Pass Cutoff (TOnly allow stuff below this)

# Calculate filter coefficients
a_hp = 1 / (1 + (2 * np.pi * f_low / fs))
a_lp = (2 * np.pi * f_high / fs) / (1 + (2 * np.pi * f_high / fs))

# --- MEMORY: High-Pass Stage (Left Wall) ---
hp_x1, hp_y1 = 0.0, 0.0
hp_x2, hp_y2 = 0.0, 0.0

# --- MEMORY: Low-Pass Stage (Right Wall) ---
lp_y1 = 0.0
lp_y2 = 0.0

# Buffer for the visualizer
plot_buffer = np.zeros(blocksize)


# FILTER LOGIC
def apply_standard_bpf(x):
    global hp_x1, hp_y1, hp_x2, hp_y2
    global lp_y1, lp_y2
    
    y = np.zeros_like(x)

    for n in range(len(x)):
        # ==========================================
        # LEFT SIDE: HIGH-PASS FILTER (Cut lows)
        # ==========================================
        out_hp1 = a_hp * (hp_y1 + x[n] - hp_x1)
        out_hp2 = a_hp * (hp_y2 + out_hp1 - hp_x2)
        
        hp_x1, hp_y1 = x[n], out_hp1
        hp_x2, hp_y2 = out_hp1, out_hp2

        # ==========================================
        # RIGHT SIDE: LOW-PASS FILTER (Cut highs)
        # ==========================================
        # Feed the High-Pass output (out_hp2) into the Low-Pass
        out_lp1 = lp_y1 + a_lp * (out_hp2 - lp_y1)
        out_lp2 = lp_y2 + a_lp * (out_lp1 - lp_y2)
        
        lp_y1 = out_lp1
        lp_y2 = out_lp2

        # Final filtered sample to speakers
        y[n] = out_lp2

    return y


# AUDIO CALLBACK
def callback(indata, outdata, frames, time_info, status):
    if status:
        print(status)

    processed_audio = apply_standard_bpf(indata[:, 0])

    # Output to both left and right speakers
    outdata[:, 0] = processed_audio
    outdata[:, 1] = processed_audio


# RUN LOOP
stream = sd.Stream(channels=(1, 2), samplerate=fs, blocksize=blocksize, callback=callback)
stream.start()


<a id='low_pass'></a>
## Low Pass Filter

In [ ]:
# FILTER PARAMETERS

# Band-Pass cutoffs
f_low = 80    
f_high = 400

# NEW: Extra Final Low-Pass Cutoff
# Setting this to 100Hz to actively cut into the 80-400Hz band
f_final_lp = 100

# Calculate filter coefficients based off selected frequencies
a_hp = 1 / (1 + (2 * np.pi * f_low / fs))
a_lp = (2 * np.pi * f_high / fs) / (1 + (2 * np.pi * f_high / fs))
a_final_lp = (2 * np.pi * f_final_lp / fs) / (1 + (2 * np.pi * f_final_lp / fs))

# State Variables for High-Pass (Stage 1 & 2)
hp_x1, hp_y1 = 0.0, 0.0
hp_x2, hp_y2 = 0.0, 0.0

# State Variables for the first Low-Pass (Stage 1 & 2)
lp_y1 = 0.0
lp_y2 = 0.0

# NEW: State Variables for Final Low-Pass (Stage 1 & 2)
final_lp_y1 = 0.0
final_lp_y2 = 0.0

# Buffer to share data
plot_buffer = np.zeros(blocksize)


# FILTER LOGIC
def apply_filters(x):
    global hp_x1, hp_y1, hp_x2, hp_y2 
    global lp_y1, lp_y2
    global final_lp_y1, final_lp_y2
    
    y = np.zeros_like(x)

    for n in range(len(x)):
        # STAGE 1: HIGH-PASS (Remove below 80Hz)
        out_hp1 = a_hp * (hp_y1 + x[n] - hp_x1)
        out_hp2 = a_hp * (hp_y2 + out_hp1 - hp_x2)
        
        hp_x1, hp_y1 = x[n], out_hp1
        hp_x2, hp_y2 = out_hp1, out_hp2

        # STAGE 2: FIRST LOW-PASS (Remove above 400Hz)
        out_lp1 = lp_y1 + a_lp * (out_hp2 - lp_y1)
        out_lp2 = lp_y2 + a_lp * (out_lp1 - lp_y2)

        lp_y1 = out_lp1
        lp_y2 = out_lp2

        # STAGE 3: FINAL LOW-PASS (Remove above 100Hz)
        out_final1 = final_lp_y1 + a_final_lp * (out_lp2 - final_lp_y1)
        out_final2 = final_lp_y2 + a_final_lp * (out_final1 - final_lp_y2)
        
        final_lp_y1 = out_final1
        final_lp_y2 = out_final2

        #run low pass again to get sharper cutoff
        out_final2 = lp_y1 + a_lp * (out_final2 - lp_y1)
        out_final2 = lp_y2 + a_lp * (out_final2 - lp_y2)

        final_lp_y1 = out_final2
        final_lp_y2 = out_final2

        # Final filtered sample
        y[n] = out_final2

    return y


# AUDIO CALLBACK
def callback(indata, outdata, frames, time_info, status):
    if status:
        print(status)

    # Process audio through all 3 stages
    processed_audio = apply_filters(indata[:, 0])

    # Output to speakers
    outdata[:, 0] = processed_audio
    outdata[:, 1] = processed_audio


print(f"Running Enhanced Band-Pass: {f_low}Hz - {f_high}Hz with Final LP at {f_final_lp}Hz")
stream = sd.Stream(channels=(1, 2), samplerate=fs, blocksize=blocksize, callback=callback)
stream.start()

Running Enhanced Band-Pass: 80Hz - 400Hz with Final LP at 100Hz


<a id='high_pass'></a>
## High Pass Filter

In [ ]:
# FILTER PARAMETERS
# Stage A: Band-Pass Filter
f_bp_low = 200    
f_bp_high = 1200

# Stage B: Final Steep High-Pass Filter
# Set to 500Hz to aggressively cut out the bottom chunk of the Band-Pass
f_final_hp = 500 

# Calculate filter coefficients
a_bp_hp = 1 / (1 + (2 * np.pi * f_bp_low / fs))
a_bp_lp = (2 * np.pi * f_bp_high / fs) / (1 + (2 * np.pi * f_bp_high / fs))
a_final_hp = 1 / (1 + (2 * np.pi * f_final_hp / fs))

# State Variables for the Band-Pass (HPF component)
bp_hp_x1, bp_hp_y1 = 0.0, 0.0
bp_hp_x2, bp_hp_y2 = 0.0, 0.0

# State Variables for the Band-Pass (LPF component)
bp_lp_y1 = 0.0
bp_lp_y2 = 0.0

# State Variables for the Final Steep High-Pass
final_hp_x1, final_hp_y1 = 0.0, 0.0
final_hp_x2, final_hp_y2 = 0.0, 0.0

# Buffer to share data with the visualizer
plot_buffer = np.zeros(blocksize)


# FILTER LOGIC
def apply_chain(x):
    # Bring in all our dedicated memory states
    global bp_hp_x1, bp_hp_y1, bp_hp_x2, bp_hp_y2 
    global bp_lp_y1, bp_lp_y2
    global final_hp_x1, final_hp_y1, final_hp_x2, final_hp_y2
    
    y = np.zeros_like(x)

    for n in range(len(x)):
        # PART 1: THE BAND-PASS FILTER
        
        # A. BPF High-Pass (Remove below 200Hz)
        out_bp_hp1 = a_bp_hp * (bp_hp_y1 + x[n] - bp_hp_x1)
        out_bp_hp2 = a_bp_hp * (bp_hp_y2 + out_bp_hp1 - bp_hp_x2)
        
        bp_hp_x1, bp_hp_y1 = x[n], out_bp_hp1
        bp_hp_x2, bp_hp_y2 = out_bp_hp1, out_bp_hp2

        # B. BPF Low-Pass (Remove above 400Hz)
        out_bp_lp1 = bp_lp_y1 + a_bp_lp * (out_bp_hp2 - bp_lp_y1)
        out_bp_lp2 = bp_lp_y2 + a_bp_lp * (out_bp_lp1 - bp_lp_y2)

        bp_lp_y1 = out_bp_lp1
        bp_lp_y2 = out_bp_lp2

        # PART 2: THE FINAL STEEP HIGH-PASS FILTER
        # This takes the output of the BPF (out_bp_lp2) and runs it 
        # through high-pass math twice for a steeper cut.
        
        out_final_hp1 = a_final_hp * (final_hp_y1 + out_bp_lp2 - final_hp_x1)
        out_final_hp2 = a_final_hp * (final_hp_y2 + out_final_hp1 - final_hp_x2)
        
        final_hp_x1, final_hp_y1 = out_bp_lp2, out_final_hp1
        final_hp_x2, final_hp_y2 = out_final_hp1, out_final_hp2

        # Final filtered sample to speakers
        y[n] = out_final_hp2

    return y

# AUDIO CALLBACK
def callback(indata, outdata, frames, time_info, status):
    if status:
        print(status)

    processed_audio = apply_chain(indata[:, 0])

    outdata[:, 0] = processed_audio
    outdata[:, 1] = processed_audio



# 6. RUN LOOP
stream = sd.Stream(channels=(1, 2), samplerate=fs, blocksize=blocksize, callback=callback)
stream.start()